In [2]:
import json
import os
import time
import string
import re
from sklearn.metrics import f1_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from transformers import AutoTokenizer
from src.model_loader import ImageQAModel
from app import ImageQASystem
from src.image_processor import load_image
from src.utils import get_device
from rich.console import Console
from dotenv import load_dotenv
load_dotenv(dotenv_path="src/.env")
gemini_api_key = os.getenv("GEMINI_API_KEY") 

/home/quynh/anaconda3/envs/zeroshotvqa/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = ImageQAModel(device=get_device(), gemini_api_key=gemini_api_key)

In [4]:

def normalize_answer(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()

    common = set(pred_tokens) & set(gt_tokens)
    num_same = len(common)
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 1 if pred_tokens == gt_tokens else 0
    if num_same == 0:
        return 0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1

In [5]:
# File paths
qa_pairs_path = "data evaluate/qa_pairs.json"
image_folder_path = "data evaluate/Images"

with open(qa_pairs_path, "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

In [7]:
count=0
for image in os.listdir(image_folder_path):
    count+=1
print("Total images: ", count)

Total images:  200


In [8]:
# Init model
device = get_device()
model = ImageQAModel(device=device,gemini_api_key=gemini_api_key)
model.load_vision_model()

/home/quynh/anaconda3/envs/zeroshotvqa/lib/python3.9/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/quynh/anaconda3/envs/zeroshotvqa/lib/python3.9/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


/home/quynh/anaconda3/envs/zeroshotvqa/lib/python3.9/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [9]:

# Metrics
exact_match_count = 0
total_questions = 0
f1_scores = []
bleu_scores = []
rouge_l_scores = []
bertscore_P = []
bertscore_R = []
bertscore_F1 = []
latencies = []

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smoothie = SmoothingFunction().method4

In [10]:

for qa_pair in qa_pairs["annotations"]:
    image_id = qa_pair["image_id"]
    question = qa_pair.get("question")
    labeled_answer = qa_pair.get("answers", [""])[0]

    if not question or not labeled_answer:
        continue

    image_path = os.path.join(image_folder_path, f"{image_id}.jpg")
    if not os.path.exists(image_path):
        continue

    pixel_values = load_image(image_path)

    start_time = time.time()
    caption = model.generate_caption(pixel_values)
    generated_answer = model.answer_question(question)
    latency = time.time() - start_time

    if not generated_answer:
        continue

    # Normalize
    labeled_answer_norm = normalize_answer(labeled_answer)
    generated_answer_norm = normalize_answer(generated_answer)

    # Exact match
    exact_match = int(generated_answer_norm == labeled_answer_norm)
    exact_match_count += exact_match

    # F1
    f1 = compute_f1(generated_answer, labeled_answer)
    f1_scores.append(f1)

    # BLEU
    bleu = sentence_bleu([labeled_answer_norm.split()], generated_answer_norm.split(), smoothing_function=smoothie)
    bleu_scores.append(bleu)

    # ROUGE-L
    rouge_l = rouge.score(labeled_answer, generated_answer)['rougeL'].fmeasure
    rouge_l_scores.append(rouge_l)

    # BERTScore
    P, R, F1 = bert_score([generated_answer], [labeled_answer], lang="vi", verbose=False)
    bertscore_P.append(P[0].item())
    bertscore_R.append(R[0].item())
    bertscore_F1.append(F1[0].item())

    latencies.append(latency)
    total_questions += 1

    print(f"Question: {question}")
    print(f"Labeled Answer: {labeled_answer}")
    print(f"Generated Answer: {generated_answer}")
    print(f"Exact Match: {exact_match}, F1: {f1:.4f}, BLEU: {bleu:.4f}, ROUGE-L: {rouge_l:.4f}, BERTScore-F1: {F1[0].item():.4f}, Latency: {latency:.2f}s")
    print("-" * 60)




Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Question: công viên này tên là gì ?
Labeled Answer: one world
Generated Answer: Đất Xanh Hiền Thượng.
Exact Match: 0, F1: 0.0000, BLEU: 0.0000, ROUGE-L: 0.0000, BERTScore-F1: 0.6510, Latency: 40.25s
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Question: công viên này thuộc nơi nào ?
Labeled Answer: dat quang riverside
Generated Answer: Không thể xác định vị trí cụ thể chỉ dựa vào hình ảnh và mô tả.  Cần thêm thông tin.
Exact Match: 0, F1: 0.0000, BLEU: 0.0000, ROUGE-L: 0.0000, BERTScore-F1: 0.6244, Latency: 36.58s
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Question: ảnh lấy nguồn từ đâu ?
Labeled Answer: dulich9 . com
Generated Answer: Không thể xác định nguồn ảnh.
Exact Match: 0, F1: 0.0000, BLEU: 0.0000, ROUGE-L: 0.0000, BERTScore-F1: 0.6457, Latency: 44.15s
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Question: quán góc dưới bên trái tên gì ?
Labeled Answer: bàu sen
Generated Answer: Bàu Sen
Exact Match: 1, F1: 1.0000, BLEU: 0.2214, ROUGE-L: 1.0000, BERTScore-F1: 0.9096, Latency: 37.14s
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


KeyboardInterrupt: 

In [11]:
# Final report
print("\nEvaluation Results:")
print(f"Total Questions: {total_questions}")
print(f"Exact Match Accuracy: {exact_match_count / total_questions:.4f}")
print(f"Average F1 Score: {sum(f1_scores)/len(f1_scores):.4f}")
print(f"Average BLEU Score: {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE-L Score: {sum(rouge_l_scores)/len(rouge_l_scores):.4f}")
print(f"Average BERTScore-F1: {sum(bertscore_F1)/len(bertscore_F1):.4f}")
print(f"Average Latency: {sum(latencies)/len(latencies):.2f} seconds")


Evaluation Results:
Total Questions: 466
Exact Match Accuracy: 0.2082
Average F1 Score: 0.4146
Average BLEU Score: 0.2242
Average ROUGE-L Score: 0.5071
Average BERTScore-F1: 0.7452
Average Latency: 83.34 seconds
